# РГР



In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# Предположим, df — твой DataFrame, а columns = ['X1', 'X2', 'X3']
# Ниже логика для выполнения пунктов 4.5 и 4.6


def process_stats_tasks(df, columns):
    results = {}

    for col in columns:
        data = df[col].values
        n = len(data)
        x_bar = np.mean(data)
        s_sq_corrected = np.var(data, ddof=1)
        s_corrected = np.sqrt(s_sq_corrected)

        # --- 4.5. Оценка моментов по сгруппированной выборке ---
        # Используем правило Стерджеса для определения количества интервалов [cite: 100]
        k = int(1 + np.floor(np.log2(n)))
        counts, bin_edges = np.histogram(data, bins=k)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        # Среднее и дисперсия по сгруппированным данным [cite: 136]
        x_bar_grouped = np.sum(counts * bin_centers) / n
        s_sq_grouped = np.sum(counts * (bin_centers - x_bar_grouped) ** 2) / (n - 1)

        # --- 4.6. Доверительные интервалы (alpha = 0.05) ---
        gamma = 0.95
        alpha = 1 - gamma

        # 1. Асимптотический ДИ для EX (по ЦПТ) [cite: 139]
        z_val = stats.norm.ppf(1 - alpha / 2)
        ci_asymptotic = (
            x_bar - z_val * (s_corrected / np.sqrt(n)),
            x_bar + z_val * (s_corrected / np.sqrt(n)),
        )

        # 2. Точные ДИ для нормального распределения [cite: 140, 145]
        # (Применяется к столбцу, который вы определили как нормальный)
        t_val = stats.t.ppf(1 - alpha / 2, df=n - 1)
        chi2_low = stats.chi2.ppf(alpha / 2, df=n - 1)
        chi2_high = stats.chi2.ppf(1 - alpha / 2, df=n - 1)

        ci_mu_exact = (
            x_bar - t_val * (s_corrected / np.sqrt(n)),
            x_bar + t_val * (s_corrected / np.sqrt(n)),
        )

        ci_sigma_sq = (
            (n - 1) * s_sq_corrected / chi2_high,
            (n - 1) * s_sq_corrected / chi2_low,
        )

        results[col] = {
            "grouped_mean": x_bar_grouped,
            "grouped_var": s_sq_grouped,
            "ci_asymptotic_ex": ci_asymptotic,
            "ci_mu_exact": ci_mu_exact,
            "ci_sigma_sq": ci_sigma_sq,
        }

        # Вывод результатов
        print(f"--- Результаты для {col} ---")
        print(f"Сгруппированное среднее: {x_bar_grouped:.4f} (исходное: {x_bar:.4f})")
        print(f"Асимптотический ДИ для EX: ({ci_asymptotic[0]:.4f}, {ci_asymptotic[1]:.4f})")
        if col == "X1":  # Замени на условие для твоего нормального столбца
            print(f"Точный ДИ для mu: {ci_mu_exact}")
            print(f"Точный ДИ для sigma^2: {ci_sigma_sq}")
        print("\n")


df = pd.read_csv('RGR1_A-2_X1-X4.csv')
data_x1 = df["X1"]
data_x2 = df["X2"]
data_x3 = df["X3"]
data_x4 = df["X4"]


process_stats_tasks(data_x1,0)
process_stats_tasks(data_x2,1)
process_stats_tasks(data_x3,2)
process_stats_tasks(data_x4,3)

--- Результаты для X1 ---
Сгруппированное среднее: 50.4106 (исходное: 50.5689)
Асимптотический ДИ для EX: (49.2779, 51.8599)
Точный ДИ для mu: (np.float64(49.270024107679674), np.float64(51.86777589232031))
Точный ДИ для sigma^2: (np.float64(71.95905089756937), np.float64(106.70257867170632))


--- Результаты для X2 ---
Сгруппированное среднее: 48.9985 (исходное: 49.0995)
Асимптотический ДИ для EX: (47.3054, 50.8936)


--- Результаты для X3 ---
Сгруппированное среднее: 83.2722 (исходное: 79.8089)
Асимптотический ДИ для EX: (71.8256, 87.7922)


--- Результаты для X4 ---
Сгруппированное среднее: 71.6338 (исходное: 71.2965)
Асимптотический ДИ для EX: (66.1129, 76.4801)





---

# Итоговый вывод по РГР №1

### 1. Идентификация законов распределения
В ходе выполнения работы был проведен статистический анализ четырех наборов данных объема $$n = 200$$. На основании графического анализа и сопоставления выборочных характеристик с теоретическими, были определены следующие модели:

* **Столбец $X_1$ — Нормальное распределение ($N_{a, \sigma}$):**
    * Среднее $$\bar{x} = 50.57$$и медиана$$\tilde{x} = 50.34$$ практически совпадают. Коэффициент асимметрии близок к нулю ($$\gamma \approx 0.27$$), что характерно для симметричного колоколообразного распределения.
* **Столбец $X_2$ — Равномерное распределение ($U_{a, b}$):**
    * Распределение симметрично ($$\gamma \approx 0.14$$), среднее $$\bar{x} = 49.10$$находится в центре размаха$$[26.8, 72.6]$$. Отсутствие ярко выраженной моды указывает на равномерный закон.
* **Столбец $X_3$ — Экспоненциальное распределение со сдвигом ($Exp_{\lambda, c}$):**
    * Наблюдается высокая положительная асимметрия ($$\gamma \approx 1.98$$) и значительный разрыв между медианой ($$63.35$$) и максимумом ($$407.0$$), что указывает на «тяжелый хвост» справа.
* **Столбец $X_4$ — Неоднородная выборка (смесь распределений):**
    * Несмотря на умеренную асимметрию, среднее ($$71.30$$) значительно удалено от медианы ($$46.76$$). Большая дисперсия ($$s^2 \approx 1392$$) и визуальная бимодальность гистограммы говорят о наличии двух групп данных.

### 2. Оценивание параметров и правила группировки
Для каждого набора данных было проведено сравнение правил определения количества интервалов ($k$):
* **Правило Стерджеса** стабильно предлагает $$k = 9$$, что является классическим подходом для данного объема выборки.
* **Правило Фридмана–Диакониса** более чувствительно к разбросу: для столбца $X_3$ с большим количеством выбросов оно предложило $$k = 20$$, что позволяет лучше детализировать структуру «хвоста» распределения.

### 3. Сравнение методов оценивания
* Для **нормального распределения** ($X_1$) оценки параметров методом моментов (ММ) и максимального правдоподобия (ММП) совпадают ($$\hat{a} = 50.57$$).
* Для **равномерного** ($X_2$) и **экспоненциального** ($X_3$) распределений метод ММП является более предпочтительным, так как он минимизирует влияние отклонений, используя границы выборки (min и max) для оценки параметров сдвига.

### 4. Доверительные интервалы и точность
Построенные доверительные интервалы с уровнем доверия $$\gamma = 0.95$$ для математического ожидания показали, что:
* Точность оценивания высока, так как стандартная ошибка среднего $$\hat{\sigma}/\sqrt{n}$$ невелика по сравнению с величиной самих значений.
* С надежностью 95% можно утверждать, что истинные значения параметров лежат в рассчитанных диапазонах.

### 5. Резюме по неоднородности ($X_4$)
Анализ столбца $X_4$ подтвердил, что стандартные статистические показатели (среднее, дисперсия) не дают адекватного представления о структуре данных при наличии бимодальности. Среднее значение $$71.30$$ не является типичным ни для одного из скоплений данных, что требует разделения выборки на подгруппы перед дальнейшим анализом.

---
